In [ ]:
#this dimension belengs to application schema
#creating statecities dimension

from pyspark.sql.session import SparkSession
import pyspark.sql.functions
import os as os

from dotenv import load_dotenv

In [ ]:
env = load_dotenv('/mnt/d/linux/project/vars/.env')

pgjar = os.getenv('POSTGRE_JAR')    
pgconn = os.getenv('POSTGRE_CONN')
pgdriver = os.getenv('PG_DRIVER')
pguser = os.getenv('POSTGRE_USER')
pgpwd = os.getenv('POSTGRE_PWD')

In [ ]:
#session 

spark = (( SparkSession
            .builder
            .appName('statecities')
            .config('spark.jars', f'{pgjar}')
            .config('spark.driver.extraClassPath', f'{pgjar}')
            .config('spark.executor.extraClassPath', f'{pgjar}')
            .getOrCreate()
        ))

In [ ]:
tmp_citiesframe =  ((spark
                        .read
                        .format('jdbc')
                        .option('url', pgconn)
                        .option('driver', pgdriver)
                        .option('user', pguser)
                        .option('password', pgpwd)
                        .option('query', 
                                """ 
                                    select 

                                        apc.cityid,
                                        apc.cityname,
                                        apc.stateprovinceid,
                                        apc.latestrecordedpopulation,
                                        apc._airbyte_ab_id,
                                        apc._airbyte_cities_hashid,
                                        cast(apc._airbyte_emitted_at as date) as _airbyte_emitted_atº,
                                        cast(apc._airbyte_normalized_at as date) as _airbyte_normalized_at

                                    from application.cities as apc

                                    left join

                                        application.cities_archive as apca
                                    on apc.cityid = apca.cityid
                                    
                                    """) 
                          )).load()

tmp_citiesframe.show(5, False)

In [32]:
tmp_stateframe = ((spark
                    .read
                    .format('jdbc')
                    .option('url', pgconn)
                    .option('driver', pgdriver)
                    .option('user', pguser)
                    .option('password', pgpwd)
                    .option('query',
                            """ 
                                select

                                    asp.countryid,
                                    asp.stateprovinceid,
                                    asp.stateprovincename,
                                    asp.salesterritory,
                                    asp.stateprovincecode,
                                    asp.latestrecordedpopulation,
                                    asp._airbyte_ab_id,
                                    asp._airbyte_stateprovinces_hashid,
                                    cast(asp._airbyte_emitted_at as date) as _airbyte_emitted_at,
                                    cast(asp._airbyte_normalized_at as date) as _airbyte_normalized_at
                                from application.stateprovinces as asp
                                
                                left join
                                    application.stateprovinces_archive as aspa
                                on asp.stateprovinceid = aspa.stateprovinceid
                            
                            """) 
                   )).load()

tmp_stateframe.show()

+---------+---------------+-----------------+--------------+-----------------+------------------------+--------------------+------------------------------+-------------------+----------------------+
|countryid|stateprovinceid|stateprovincename|salesterritory|stateprovincecode|latestrecordedpopulation|      _airbyte_ab_id|_airbyte_stateprovinces_hashid|_airbyte_emitted_at|_airbyte_normalized_at|
+---------+---------------+-----------------+--------------+-----------------+------------------------+--------------------+------------------------------+-------------------+----------------------+
|      230|              1|          Alabama|     Southeast|               AL|                 5437278|19e57a2d-0cf8-48e...|          0b01e222063d095da...|         2023-07-17|            2023-07-17|
|      230|              4|         Arkansas|     Southeast|               AR|                 3077747|dbd60e2e-6eda-42b...|          d91c7928c81a5dd96...|         2023-07-17|            2023-07-17|
|    